**In this notebook we will create the Scikit-Learn K nearest Neighbour Classifier Class From Scratch**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error , accuracy_score
from sklearn.model_selection import cross_val_score

In [2]:
df = pd.read_csv("Social_Network_Ads.csv")
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [3]:
df.dtypes

User ID            int64
Gender               str
Age                int64
EstimatedSalary    int64
Purchased          int64
dtype: object

Raw data (X, y)
       ↓
train_test_split()
       ↓
 ┌─────────────┐      ┌──────────────┐
 │   X_train   │      │   X_test     │  ← lock this away
 │   y_train   │      │   y_test     │
 └─────────────┘      └──────────────┘
       ↓
Build Pipeline (OrdinalEncoder + StandardScaler + KNN)
       ↓
cross_val_score(pipeline, X_train, y_train)  ← tune k, compare models
       ↓
pipeline.fit(X_train, y_train)   ← final fit on ALL training data
       ↓
pipeline.score(X_test, y_test)   ← ONE final honest evaluation

In [4]:
X = df.iloc[:,1:4] # Features dataset
y = df.iloc[:,-1]  # Target Column Dataset

In [5]:
print(X.shape)
print(y.shape)

(400, 3)
(400,)


In [11]:
X.select_dtypes(include=["object", "str"]).columns.tolist()

['Gender']

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier

# ─────────────────────────────────────────────
# 1. IDENTIFY CATEGORICAL AND NUMERIC COLUMNS
# ─────────────────────────────────────────────
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols   = X.select_dtypes(include=["number"]).columns.tolist()

print(f"Categorical columns : {categorical_cols}")
print(f"Numerical columns   : {numerical_cols}")

# ─────────────────────────────────────────────
# 2. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # preserves class proportions
)

print(f"\nTraining samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")

# ─────────────────────────────────────────────
# 3. BUILD PREPROCESSOR + PIPELINE
# ─────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
    ("scaler",  StandardScaler(), numerical_cols)
])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())
])

# ─────────────────────────────────────────────
# 4. FIND BEST K USING CROSS-VALIDATION
# ─────────────────────────────────────────────
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

k_values  = range(1, 31)          # test k from 1 to 30
cv_means  = []
cv_stds   = []

print("\n--- Cross-Validation Results ---")
print(f"{'k':<6} {'Mean Accuracy':<18} {'Std'}")
print("-" * 36)

for k in k_values:
    pipeline.set_params(knn__n_neighbors=k)

    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv_strategy,
        scoring="accuracy"
    )

    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f"{k:<6} {scores.mean():.4f}            ± {scores.std():.4f}")

# ─────────────────────────────────────────────
# 5. PICK BEST K
# ─────────────────────────────────────────────
best_index    = np.argmax(cv_means)
best_k        = list(k_values)[best_index]
best_cv_score = cv_means[best_index]
best_cv_std   = cv_stds[best_index]

print(f"\nBest k            : {best_k}")
print(f"Best CV Accuracy  : {best_cv_score:.4f} ± {best_cv_std:.4f}")

# ─────────────────────────────────────────────
# 6. FINAL FIT ON FULL TRAINING DATA WITH BEST K
# ─────────────────────────────────────────────
pipeline.set_params(knn__n_neighbors=best_k)
pipeline.fit(X_train, y_train)

print(f"\nPipeline fitted with k = {best_k}")

# ─────────────────────────────────────────────
# 7. EVALUATE ON HELD-OUT TEST SET (DONE ONCE)
# ─────────────────────────────────────────────
test_accuracy = pipeline.score(X_test, y_test)
print(f"Test Set Accuracy : {test_accuracy:.4f}")

C:\Users\Pradhuman\AppData\Local\Temp\ipykernel_16796\950789304.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


Categorical columns : ['Gender']
Numerical columns   : ['Age', 'EstimatedSalary']

Training samples : 320
Test samples     : 80

--- Cross-Validation Results ---
k      Mean Accuracy      Std
------------------------------------
1      0.8781            ± 0.0230
2      0.8688            ± 0.0351
3      0.8969            ± 0.0212
4      0.8906            ± 0.0221
5      0.9062            ± 0.0140
6      0.9031            ± 0.0117
7      0.9062            ± 0.0099
8      0.9125            ± 0.0159
9      0.9094            ± 0.0153
10     0.9031            ± 0.0207
11     0.9000            ± 0.0077
12     0.8938            ± 0.0182
13     0.8906            ± 0.0140
14     0.8844            ± 0.0254
15     0.8875            ± 0.0250
16     0.8844            ± 0.0337
17     0.8906            ± 0.0221
18     0.8812            ± 0.0234
19     0.8844            ± 0.0290
20     0.8906            ± 0.0221
21     0.8906            ± 0.0296
22     0.8938            ± 0.0269
23     0.8875          